# 02 — Clean (step 1: resolve CLAUDE.md -> AGENTS.md)

The unit is the **repo**. Each repo gets one `content_resolved` and a `resolved_from` label. **`CLAUDE.md` takes priority; if the repo has no `CLAUDE.md`, use its `AGENTS.md`.**
- `CLAUDE.md` exists and is substantive -> use it.
- `CLAUDE.md` is a pure pointer (symlink to AGENTS.md, or short and mentions AGENTS.md) -> use `AGENTS.md`.
- `CLAUDE.md` imports `@AGENTS.md` but has its own content -> `CLAUDE.md` text + `AGENTS.md` text, labelled `CLAUDE.md+AGENTS.md`.
- No `CLAUDE.md`, only `AGENTS.md` -> use `AGENTS.md`.
- Pointer but the target text is missing -> `unresolved` (kept, raw text unchanged).

One hop only; other `@imports` are not followed. `content_raw` is never overwritten. Dedup and secret stripping come in later steps of this notebook.

In [1]:
import re
from pathlib import Path
import pandas as pd

IN_PATH = Path("data/raw/pilot_50.parquet")
OUT_PATH = Path("data/interim/resolved.parquet")
POINTER_MAX_CHARS = 500          # a CLAUDE.md shorter than this that mentions AGENTS.md is treated as a pointer
IMPORT_RE = re.compile(r"(^|\s)@\.?/?AGENTS\.md\b", re.I)
MENTION_RE = re.compile(r"AGENTS\.md", re.I)

raw = pd.read_parquet(IN_PATH)
print(raw.shape)

(50, 25)


## 1. One row per repo

In [2]:
repos = raw.drop_duplicates("full_name").copy()    # one row per repo; claude_text / agents_text come straight from the scrape
repos["claude_symlink_to_agents"] = repos["claude_symlink_target"].fillna("").str.contains("AGENTS.md", case=False)
print(len(repos), "repos")

50 repos


## 2. Detect pointers and resolve

In [3]:
def resolve(r):
    c = None if pd.isna(r.claude_text) else r.claude_text
    a = None if pd.isna(r.agents_text) else r.agents_text
    if c is None and not r.claude_symlink_to_agents:
        return pd.Series({"is_redirect": False, "resolved_from": "AGENTS.md", "content_resolved": a})
    body = (c or "").strip()
    is_import = bool(IMPORT_RE.search(body))
    pure_pointer = r.claude_symlink_to_agents or (len(body) < POINTER_MAX_CHARS and bool(MENTION_RE.search(body)))
    if not (pure_pointer or is_import):
        return pd.Series({"is_redirect": False, "resolved_from": "CLAUDE.md", "content_resolved": c})
    if a is None:
        return pd.Series({"is_redirect": True, "resolved_from": "unresolved", "content_resolved": c})
    if pure_pointer:
        return pd.Series({"is_redirect": True, "resolved_from": "AGENTS.md", "content_resolved": a})
    return pd.Series({"is_redirect": True, "resolved_from": "CLAUDE.md+AGENTS.md", "content_resolved": c + "\n\n" + a})

repos[["is_redirect", "resolved_from", "content_resolved"]] = repos.apply(resolve, axis=1)
repos["n_chars_resolved"] = repos["content_resolved"].str.len()
repos["no_claude_md"] = ~repos["has_claude_md"]
print(repos["resolved_from"].value_counts())
print("redirects:", repos["is_redirect"].sum(), "| AGENTS.md-only repos:", repos["no_claude_md"].sum())

resolved_from
CLAUDE.md              32
AGENTS.md              16
CLAUDE.md+AGENTS.md     2
Name: count, dtype: int64
redirects: 6 | AGENTS.md-only repos: 12


## 3. Eyeball the redirects and the short files

In [4]:
repos.loc[repos.is_redirect | (repos.n_chars_resolved < 400),
          ["full_name", "primary_file", "has_both", "is_redirect", "resolved_from", "n_chars", "n_chars_resolved"]]

,full_name,primary_file,has_both,is_redirect,resolved_from,n_chars,n_chars_resolved
3,NachoSoto/AsyncImageView,AGENTS.md,False,False,AGENTS.md,202,202
5,huang-sh/PiX,AGENTS.md,False,False,AGENTS.md,353,353
15,ruby-no-kai/regional.rubykaigi.org,CLAUDE.md,True,True,AGENTS.md,11,1171
26,cdavid817/vanehub-ai,CLAUDE.md,True,True,CLAUDE.md+AGENTS.md,609,7643
29,sveneisenschmidt/n8n-openai-bridge,CLAUDE.md,True,True,AGENTS.md,11,8145
32,darksworm/argonaut,CLAUDE.md,False,False,CLAUDE.md,265,265
34,adrianiftode/Assertions.Web,CLAUDE.md,False,False,CLAUDE.md,140,140
37,rmordechay/glsl-plugin-idea,CLAUDE.md,True,False,CLAUDE.md,59,59
39,cisagov/ScubaConnect,CLAUDE.md,True,True,AGENTS.md,10,2586
44,delphix/delphix-platform,CLAUDE.md,False,False,CLAUDE.md,275,275


## 4. Checkpoint

In [5]:
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
repos.drop(columns=["claude_text", "agents_text"]).to_parquet(OUT_PATH, index=False)
print("saved", OUT_PATH, repos.shape)

saved data/interim/resolved.parquet (50, 31)
